# Compress Any LLM's Memory in 3 Lines

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/codepawl/turboquant-torch/blob/main/examples/01_quickstart.ipynb)

This notebook shows `turboquant.wrap()` in action: generate text with automatic KV cache compression, compare memory and output quality.

## Install

In [ ]:
!pip install -q "turboquant-torch[hf]"

## Load Model

SmolLM2-135M is small enough for Colab free tier (CPU).

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "HuggingFaceTB/SmolLM2-135M"
PROMPT = "The future of artificial intelligence will"

model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model.eval()

inputs = tokenizer(PROMPT, return_tensors="pt")
print(f"Model: {MODEL}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")
print(f"Prompt: {PROMPT!r} ({inputs['input_ids'].shape[1]} tokens)")

## Generate WITHOUT Compression

In [ ]:
with torch.no_grad():
    baseline_out = model.generate(
        **inputs, max_new_tokens=50, do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

baseline_text = tokenizer.decode(baseline_out[0], skip_special_tokens=True)
print(f"Output: {baseline_text!r}")

# Show KV cache memory
from turboquant import TurboQuantKVCache
from turboquant.compat import detect_model_kv_info

info = detect_model_kv_info(model)
seq_len = baseline_out.shape[1]
cache = TurboQuantKVCache(head_dim=info.head_dim, bit_width=3, residual_length=0)
orig_mb, comp_mb, ratio = cache.memory_savings(1, info.num_kv_heads, seq_len)
print(f"KV cache memory: {orig_mb * info.n_layers:.1f} MB (fp32)")

## Generate WITH TurboQuant — 3 Lines

In [ ]:
import turboquant

wrapped = turboquant.wrap(model, verbose=True)

with torch.no_grad():
    tq_out = wrapped.generate(
        **inputs, max_new_tokens=50, do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

tq_text = tokenizer.decode(tq_out[0], skip_special_tokens=True)
print(f"Output: {tq_text!r}")
print(f"KV cache memory: {comp_mb * info.n_layers:.1f} MB (3-bit) — {ratio:.1f}x smaller")

## Compare Results

In [ ]:
baseline_tokens = baseline_out[0].tolist()
tq_tokens = tq_out[0].tolist()
min_len = min(len(baseline_tokens), len(tq_tokens))
match = sum(baseline_tokens[i] == tq_tokens[i] for i in range(min_len))
total = len(baseline_tokens)

print(f"Token match:  {match}/{total} ({100 * match / total:.0f}%)")
print(f"Memory saved: {orig_mb * info.n_layers:.1f} MB → {comp_mb * info.n_layers:.1f} MB ({ratio:.1f}x)")
print()
print("Baseline:", baseline_text[:80])
print("TurboQ: ", tq_text[:80])

## Configuration Options

```python
# Basic (auto-detect everything):
model = turboquant.wrap(model)

# With options:
model = turboquant.wrap(
    model,
    bit_width=3,           # 2, 3, or 4
    residual_length=128,   # sliding window
    n_outlier_channels=8,  # outlier routing
    verbose=True,          # print stats
)

# Or use the cache directly:
from turboquant import TurboQuantDynamicCache
cache = TurboQuantDynamicCache.from_model(model)
output = model.generate(**inputs, past_key_values=cache)
```